# 03 — Model: X-learner CATE + bootstrap CI

**Project:** Training ROI Predictor (H08)

Two GradientBoostingRegressors as outcome models on each arm; two more as effect models on the imputed individual treatment effects; a logistic propensity head blends the two effect predictions. Bootstrap CIs come from re-fitting the whole stack on resampled training data.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from training_roi.data import PROCESSED, load_panel, make_training_artifacts
from training_roi import models
from training_roi.models import fit_x_learner, save

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

In [ ]:
PARQUET = PROCESSED / 'training_outcomes.parquet'
df = load_panel() if PARQUET.exists() else make_training_artifacts()
df_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['treatment'])
df_train.shape, df_test.shape

## 1. Fit the X-learner

In [ ]:
xl = fit_x_learner(df_train)
cate_train = xl.predict_cate(df_train)
cate_test = xl.predict_cate(df_test)
print(f'mean predicted CATE (train): {cate_train.mean():.3f}')
print(f'mean predicted CATE (test):  {cate_test.mean():.3f}')
print(f'true mean CATE (test):       {df_test["true_cate"].mean():.3f}')
print(f'MAE vs ground-truth (test):  {mean_absolute_error(df_test["true_cate"], cate_test):.3f}')

## 2. Predicted CATE distribution + truth overlay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
sns.kdeplot(df_test['true_cate'], label='ground truth', ax=ax)
sns.kdeplot(cate_test, label='X-learner', ax=ax)
ax.set_title('Test-set CATE distribution: truth vs prediction')
ax.legend()
plt.tight_layout(); plt.show()

## 3. Predicted vs true CATE — scatter

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(df_test['true_cate'], cate_test, alpha=0.4, s=14, color='#3a7ca5')
lo, hi = df_test['true_cate'].min(), df_test['true_cate'].max()
ax.plot([lo, hi], [lo, hi], color='red', ls='--')
ax.set_xlabel('true CATE'); ax.set_ylabel('predicted CATE')
ax.set_title('X-learner: predicted vs ground truth')
plt.tight_layout(); plt.show()

## 4. CATE breakdown by training kind

In [ ]:
from training_roi.data import TRAINING_TYPES
kind_lookup = {tid: kind for tid, kind in TRAINING_TYPES}
df_test = df_test.assign(pred_cate=cate_test, training_kind=df_test['training_id'].map(kind_lookup))
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.boxplot(data=df_test, x='training_kind', y='pred_cate', ax=ax)
ax.set_title('Predicted CATE by training kind')
plt.tight_layout(); plt.show()

## 5. Bootstrap CI for a sample employee

In [ ]:
demo = df_test.iloc[:5].copy()
point, lo, hi = xl.predict_cate_with_ci(demo, df_train, n_boot=20, alpha=0.10, seed=0)
demo_out = demo.assign(point=point.round(3), lo=lo.round(3), hi=hi.round(3))
demo_out[['employee_id', 'training_id', 'pre_perf', 'role_level', 'point', 'lo', 'hi', 'true_cate']]

## 6. Per-employee CI width distribution

If most CIs are narrow, the per-employee point estimate is actionable; if they are wide, the dashboard should default to the band view.

In [ ]:
rng = np.random.default_rng(0)
sample = df_test.sample(80, random_state=0)
point, lo, hi = xl.predict_cate_with_ci(sample, df_train, n_boot=20, alpha=0.10, seed=1)
widths = hi - lo
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(widths, bins=20, color='#9c6644', edgecolor='white')
ax.set_xlabel('90% CI width')
ax.set_title('Per-employee CI width distribution')
plt.tight_layout(); plt.show()

## 7. Persist artefact

In [ ]:
save(xl, 'x_learner.joblib')
print('saved x_learner.joblib')

## 8. What this notebook commits to

The persisted X-learner is now what the API loads. `04_eval.ipynb` produces the headline scorecard, the population-uplift histogram, and the slice-level CATE breakdown.